# Natural Language Processing with Disaster Tweets

This notebook keeps the original educational focus of the project while fixing the earlier preprocessing bug, removing avoidable data leakage, and making the modeling workflow reproducible.

**What this notebook now does**
- Cleans train and test text consistently
- Assigns processed text back to `train_df` and `test_df` **positionally**
- Splits the labeled data into train/validation sets with a fixed random seed
- Fits vectorization artifacts on the **training split only**
- Compares a majority baseline, a TF-IDF + Logistic Regression baseline, and a PyTorch BiLSTM
- Reports validation accuracy, precision, recall, F1, and confusion matrices
- Prints representative false positives and false negatives for error analysis

> The Kaggle competition files (`train.csv` and `test.csv`) are intentionally not committed to this repository. If they are missing, the notebook will skip modeling cells and print setup instructions instead of failing.


In [ ]:
import os
import random
import re
import sys
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import nltk
import torch
import torch.nn as nn
import torch.optim as optim
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, precision_score, recall_score
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, TensorDataset

SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
if hasattr(torch.backends, 'cudnn'):
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
try:
    torch.use_deterministic_algorithms(True)
except Exception as exc:
    print(f'Deterministic algorithm mode could not be fully enabled: {exc}')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
REPO_ROOT = Path.cwd()
TRAIN_PATH = REPO_ROOT / 'train.csv'
TEST_PATH = REPO_ROOT / 'test.csv'

print(f'Python: {sys.version.split()[0]}')
print(f'pandas: {pd.__version__}')
print(f'numpy: {np.__version__}')
print(f'torch: {torch.__version__}')
print(f'Device: {DEVICE}')
print(f'Repository root: {REPO_ROOT}')


In [ ]:
def ensure_nltk_resource(resource_path: str, download_name: str) -> bool:
    try:
        nltk.data.find(resource_path)
        return True
    except LookupError:
        try:
            return bool(nltk.download(download_name, quiet=True))
        except Exception as exc:
            print(f'Could not download {download_name}: {exc}')
            return False

stopwords_available = ensure_nltk_resource('corpora/stopwords', 'stopwords')
wordnet_available = ensure_nltk_resource('corpora/wordnet', 'wordnet')
omw_available = ensure_nltk_resource('corpora/omw-1.4', 'omw-1.4') if wordnet_available else False

if stopwords_available:
    stop_words = set(stopwords.words('english'))
else:
    stop_words = set(ENGLISH_STOP_WORDS)
    print('Falling back to scikit-learn English stop words because the NLTK stopwords corpus is unavailable.')

lemmatizer = WordNetLemmatizer()

def safe_lemmatize(token: str) -> str:
    if wordnet_available and omw_available:
        try:
            return lemmatizer.lemmatize(token)
        except LookupError:
            pass
    return token

if wordnet_available and omw_available:
    print('NLTK stopwords and WordNet resources are ready.')
else:
    print('WordNet resources are unavailable, so preprocessing will skip lemmatization instead of failing.')


## Load the dataset

This repository does **not** include the Kaggle data files. To run the full workflow locally, place `train.csv` and `test.csv` in the repository root before executing the notebook.


In [ ]:
DATA_AVAILABLE = TRAIN_PATH.exists() and TEST_PATH.exists()

if DATA_AVAILABLE:
    train_df = pd.read_csv(TRAIN_PATH)
    test_df = pd.read_csv(TEST_PATH)
    print(f'train_df shape: {train_df.shape}')
    print(f'test_df shape: {test_df.shape}')
else:
    train_df = None
    test_df = None
    print('train.csv and test.csv were not found in the repository root.')
    print('Add the Kaggle competition files locally to run EDA, baselines, and BiLSTM training.')


In [ ]:
if DATA_AVAILABLE:
    print('\nTraining columns:')
    print(train_df.columns.tolist())

    print('\nMissing values in training data:')
    print(train_df.isna().sum())

    target_distribution = train_df['target'].value_counts(normalize=True).sort_index().rename(index={0: 'not_disaster', 1: 'disaster'})
    print('\nTarget distribution:')
    print(target_distribution)

    plt.figure(figsize=(6, 4))
    sns.countplot(data=train_df, x='target')
    plt.title('Training Label Distribution')
    plt.xlabel('Target')
    plt.ylabel('Count')
    plt.xticks([0, 1], ['Not disaster', 'Disaster'])
    plt.tight_layout()
    plt.show()
else:
    print('Skipping exploratory analysis because the dataset files are not available.')


## Text preprocessing

The cleaning function is deterministic and can safely be applied to both train and test text. The important leakage boundary comes later: learned artifacts such as the TF-IDF vectorizer and neural vocabulary are fit on the **training split only**.


In [ ]:
def clean_text(text: str) -> str:
    text = str(text).lower()
    text = re.sub(r'https?://\S+|www\.\S+', ' ', text)
    text = re.sub(r'<.*?>', ' ', text)
    text = re.sub(r'@\w+', ' ', text)
    text = text.replace('#', ' ')
    text = re.sub(r'[^a-z\s]', ' ', text)
    text = re.sub(r'\b\w*\d\w*\b', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text


def tokenize_and_lemmatize(text: str) -> str:
    tokens = []
    for token in clean_text(text).split():
        if token in stop_words:
            continue
        lemma = safe_lemmatize(token)
        if lemma:
            tokens.append(lemma)
    return ' '.join(tokens)


if DATA_AVAILABLE:
    combined_df = pd.concat([train_df.drop(columns=['target']), test_df], ignore_index=True)
    combined_df['processed_text'] = combined_df['text'].map(tokenize_and_lemmatize)

    train_df['processed_text'] = combined_df.iloc[: len(train_df)]['processed_text'].to_numpy()
    test_df['processed_text'] = combined_df.iloc[len(train_df) :]['processed_text'].to_numpy()

    train_df['processed_text'] = train_df['processed_text'].fillna('')
    test_df['processed_text'] = test_df['processed_text'].fillna('')

    print('Empty processed training rows:', int((train_df['processed_text'] == '').sum()))
    print('Empty processed test rows:', int((test_df['processed_text'] == '').sum()))
    print('\nProcessed training sample:')
    print(train_df[['text', 'processed_text']].head())
else:
    print('Skipping preprocessing because the dataset files are not available.')


In [ ]:
def evaluate_predictions(model_name: str, y_true, y_pred):
    return {
        'model': model_name,
        'accuracy': accuracy_score(y_true, y_pred),
        'precision': precision_score(y_true, y_pred, zero_division=0),
        'recall': recall_score(y_true, y_pred, zero_division=0),
        'f1': f1_score(y_true, y_pred, zero_division=0),
        'confusion_matrix': confusion_matrix(y_true, y_pred),
    }


results = []
validation_predictions = {}
history_df = None

if DATA_AVAILABLE:
    modeling_df = train_df[['text', 'processed_text', 'target']].copy()
    train_split_df, val_df = train_test_split(
        modeling_df,
        test_size=0.2,
        random_state=SEED,
        stratify=modeling_df['target'],
    )
    train_split_df = train_split_df.reset_index(drop=True)
    val_df = val_df.reset_index(drop=True)

    majority_class = int(train_split_df['target'].mode().iat[0])
    majority_pred = np.full(len(val_df), majority_class)
    majority_result = evaluate_predictions('Majority class baseline', val_df['target'], majority_pred)
    results.append(majority_result)
    validation_predictions['Majority class baseline'] = majority_pred

    tfidf_vectorizer = TfidfVectorizer(ngram_range=(1, 2), min_df=2)
    X_train_tfidf = tfidf_vectorizer.fit_transform(train_split_df['processed_text'])
    X_val_tfidf = tfidf_vectorizer.transform(val_df['processed_text'])
    X_test_tfidf = tfidf_vectorizer.transform(test_df['processed_text'])

    logistic_regression = LogisticRegression(max_iter=1000, solver='liblinear', random_state=SEED)
    logistic_regression.fit(X_train_tfidf, train_split_df['target'])
    val_pred_logreg = logistic_regression.predict(X_val_tfidf)
    logreg_result = evaluate_predictions('TF-IDF + Logistic Regression', val_df['target'], val_pred_logreg)
    results.append(logreg_result)
    validation_predictions['TF-IDF + Logistic Regression'] = val_pred_logreg

    print('Validation metrics after the non-neural baselines:')
    print(pd.DataFrame([{k: v for k, v in result.items() if k != 'confusion_matrix'} for result in results]).round(4))
    print(f'\nTF-IDF feature count: {X_train_tfidf.shape[1]}')
    print(f'Test TF-IDF matrix shape (transformed with training-only vectorizer): {X_test_tfidf.shape}')
else:
    print('Skipping train/validation split and scikit-learn baselines because the dataset files are not available.')


## Neural baseline: BiLSTM

The BiLSTM uses a vocabulary built from the **training split only**. Validation and test sequences are transformed with the learned training vocabulary so that no information from validation or test text leaks into the neural input representation.


In [ ]:
def build_vocab(texts, min_freq: int = 1, pad_token: str = '<pad>', unk_token: str = '<unk>'):
    counter = Counter()
    for text in texts:
        counter.update(text.split())

    word_to_idx = {pad_token: 0, unk_token: 1}
    for word, count in counter.items():
        if count >= min_freq:
            word_to_idx.setdefault(word, len(word_to_idx))

    idx_to_word = {idx: word for word, idx in word_to_idx.items()}
    return word_to_idx, idx_to_word


def text_to_sequence(text: str, word_to_idx, max_length: int):
    tokens = text.split()
    sequence = [word_to_idx.get(token, word_to_idx['<unk>']) for token in tokens]
    sequence = sequence[:max_length]
    padding = [word_to_idx['<pad>']] * max(0, max_length - len(sequence))
    return sequence + padding


class BiLSTMClassifier(nn.Module):
    def __init__(self, vocab_size: int, embedding_dim: int = 100, hidden_dim: int = 128, dropout: float = 0.3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.lstm = nn.LSTM(
            embedding_dim,
            hidden_dim,
            batch_first=True,
            bidirectional=True,
        )
        self.dropout = nn.Dropout(dropout)
        self.fc1 = nn.Linear(hidden_dim * 2, 64)
        self.fc2 = nn.Linear(64, 1)

    def forward(self, text):
        embedded = self.embedding(text)
        outputs, _ = self.lstm(embedded)
        pooled, _ = torch.max(outputs, dim=1)
        hidden = torch.relu(self.fc1(self.dropout(pooled)))
        return self.fc2(self.dropout(hidden)).squeeze(1)


def run_epoch(model, dataloader, criterion, optimizer=None):
    is_training = optimizer is not None
    model.train() if is_training else model.eval()

    total_loss = 0.0
    all_predictions = []
    all_targets = []

    for batch in dataloader:
        features = batch[0].to(DEVICE)
        targets = batch[1].to(DEVICE)

        with torch.set_grad_enabled(is_training):
            logits = model(features)
            loss = criterion(logits, targets)
            predictions = (torch.sigmoid(logits) >= 0.5).float()

            if is_training:
                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()

        total_loss += loss.item() * len(features)
        all_predictions.extend(predictions.detach().cpu().numpy())
        all_targets.extend(targets.detach().cpu().numpy())

    average_loss = total_loss / len(dataloader.dataset)
    average_accuracy = accuracy_score(all_targets, all_predictions)
    return average_loss, average_accuracy


def predict_with_model(model, dataloader):
    model.eval()
    predictions = []
    with torch.no_grad():
        for batch in dataloader:
            features = batch[0].to(DEVICE)
            logits = model(features)
            batch_predictions = (torch.sigmoid(logits) >= 0.5).long().cpu().numpy()
            predictions.extend(batch_predictions)
    return np.array(predictions)


In [ ]:
if DATA_AVAILABLE:
    word_to_idx, idx_to_word = build_vocab(train_split_df['processed_text'])
    max_length = max(5, int(np.ceil(train_split_df['processed_text'].str.split().str.len().quantile(0.95))))

    X_train_seq = np.array([text_to_sequence(text, word_to_idx, max_length) for text in train_split_df['processed_text']])
    X_val_seq = np.array([text_to_sequence(text, word_to_idx, max_length) for text in val_df['processed_text']])
    X_test_seq = np.array([text_to_sequence(text, word_to_idx, max_length) for text in test_df['processed_text']])

    y_train = train_split_df['target'].to_numpy(dtype=np.float32)
    y_val = val_df['target'].to_numpy(dtype=np.float32)

    train_dataset = TensorDataset(torch.tensor(X_train_seq, dtype=torch.long), torch.tensor(y_train, dtype=torch.float32))
    val_dataset = TensorDataset(torch.tensor(X_val_seq, dtype=torch.long), torch.tensor(y_val, dtype=torch.float32))
    test_dataset = TensorDataset(torch.tensor(X_test_seq, dtype=torch.long))

    generator = torch.Generator().manual_seed(SEED)
    train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True, generator=generator)
    val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

    print(f'Vocabulary size learned from training split: {len(word_to_idx)}')
    print(f'Max sequence length from training split (95th percentile): {max_length}')
    print(f'Test sequence tensor shape (transformed with training-only vocabulary): {X_test_seq.shape}')

    model = BiLSTMClassifier(vocab_size=len(word_to_idx)).to(DEVICE)
    criterion = nn.BCEWithLogitsLoss()
    optimizer = optim.Adam(model.parameters(), lr=1e-3)

    best_state_dict = None
    best_val_loss = float('inf')
    patience = 2
    patience_counter = 0
    history = []

    for epoch in range(1, 9):
        train_loss, train_accuracy = run_epoch(model, train_loader, criterion, optimizer)
        val_loss, val_accuracy = run_epoch(model, val_loader, criterion)
        history.append({
            'epoch': epoch,
            'train_loss': train_loss,
            'val_loss': val_loss,
            'train_accuracy': train_accuracy,
            'val_accuracy': val_accuracy,
        })
        print(
            f'Epoch {epoch:02d} | '
            f'train_loss={train_loss:.4f} | train_acc={train_accuracy:.4f} | '
            f'val_loss={val_loss:.4f} | val_acc={val_accuracy:.4f}'
        )

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state_dict = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print('Early stopping triggered.')
                break

    if best_state_dict is not None:
        model.load_state_dict(best_state_dict)

    history_df = pd.DataFrame(history)
    val_pred_bilstm = predict_with_model(model, val_loader)
    bilstm_result = evaluate_predictions('BiLSTM', val_df['target'], val_pred_bilstm)
    results.append(bilstm_result)
    validation_predictions['BiLSTM'] = val_pred_bilstm

    print('\nValidation metrics including the BiLSTM:')
    print(pd.DataFrame([{k: v for k, v in result.items() if k != 'confusion_matrix'} for result in results]).round(4))
else:
    print('Skipping neural modeling because the dataset files are not available.')


In [ ]:
if DATA_AVAILABLE:
    results_df = pd.DataFrame([{k: v for k, v in result.items() if k != 'confusion_matrix'} for result in results]).sort_values('f1', ascending=False)
    print(results_df.round(4))

    fig, axes = plt.subplots(1, len(results), figsize=(6 * len(results), 5))
    axes = np.atleast_1d(axes)
    for ax, result in zip(axes, results):
        sns.heatmap(result['confusion_matrix'], annot=True, fmt='d', cmap='Blues', cbar=False, ax=ax)
        ax.set_title(result['model'])
        ax.set_xlabel('Predicted label')
        ax.set_ylabel('True label')
    plt.tight_layout()
    plt.show()

    if history_df is not None and not history_df.empty:
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))
        axes[0].plot(history_df['epoch'], history_df['train_loss'], label='Train loss')
        axes[0].plot(history_df['epoch'], history_df['val_loss'], label='Validation loss')
        axes[0].set_title('BiLSTM loss by epoch')
        axes[0].set_xlabel('Epoch')
        axes[0].set_ylabel('Loss')
        axes[0].legend()

        axes[1].plot(history_df['epoch'], history_df['train_accuracy'], label='Train accuracy')
        axes[1].plot(history_df['epoch'], history_df['val_accuracy'], label='Validation accuracy')
        axes[1].set_title('BiLSTM accuracy by epoch')
        axes[1].set_xlabel('Epoch')
        axes[1].set_ylabel('Accuracy')
        axes[1].legend()

        plt.tight_layout()
        plt.show()
else:
    print('Skipping comparison plots because the dataset files are not available.')


In [ ]:
def show_representative_errors(frame: pd.DataFrame, predictions: np.ndarray, model_name: str, n_examples: int = 5):
    analysis_df = frame[['text', 'processed_text', 'target']].copy().reset_index(drop=True)
    analysis_df['prediction'] = predictions

    false_positives = analysis_df[(analysis_df['target'] == 0) & (analysis_df['prediction'] == 1)]
    false_negatives = analysis_df[(analysis_df['target'] == 1) & (analysis_df['prediction'] == 0)]

    print(f'\nRepresentative false positives for {model_name}:')
    if false_positives.empty:
        print('None in this validation split.')
    else:
        print(false_positives[['text', 'processed_text']].head(n_examples).to_string(index=False))

    print(f'\nRepresentative false negatives for {model_name}:')
    if false_negatives.empty:
        print('None in this validation split.')
    else:
        print(false_negatives[['text', 'processed_text']].head(n_examples).to_string(index=False))


if DATA_AVAILABLE:
    candidate_models = [
        name for name in ['BiLSTM', 'TF-IDF + Logistic Regression']
        if name in validation_predictions
    ]
    best_model_name = max(
        candidate_models,
        key=lambda name: next(result['f1'] for result in results if result['model'] == name),
    )
    print(f'Using {best_model_name} for qualitative error analysis.')
    show_representative_errors(val_df, validation_predictions[best_model_name], best_model_name)
else:
    print('Skipping error analysis because the dataset files are not available.')


## Interpretation

Disaster language is genuinely ambiguous. False positives often come from figurative expressions such as *"this exam was a disaster"* or *"my phone is on fire"*, while false negatives can occur when a tweet describes a real event without the most obvious disaster keywords. That is why the notebook compares multiple baselines, reports precision/recall alongside accuracy, and inspects specific mistakes instead of relying on a single headline metric.
